# 04 — Revenue forecasting (the honest ML bet)

Why here and not K11/K12: the monthly series grew ~30× with flat noise — trend + stability is what forecasting needs.

**Protocol (leak-proof):** daily revenue series (731 days) → expanding-window rolling origin over the **last 90 days**, refit every 7 days, **one-step-ahead** predictions. Metric = MAPE (the business cost of being wrong), RMSE/MAE alongside. A model ships only if it beats seasonal-naive by a margin worth its complexity.

Lineup: naive · seasonal-naive(7) · Holt-Winters (additive, weekly season) · RandomForest · GradientBoosting · MLP(100→50). DL gets the same fair shot — on 731 points it should lose, and we'll say so if it does.

*Needs (experiment-scoped):* `uv pip install --index-url https://download.pytorch.org/whl/cpu torch` + `uv pip install prophet`.


In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

CSV = Path("../data/amazon-e-commerce/amazon_ecommerce_1M.csv")
rev = defaultdict(float)
with open(CSV, newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        rev[row["purchase_date"]] += float(row["final_price"])
days = sorted(rev)
y = np.array([rev[d] for d in days])
print(f"days: {len(y)} ({days[0]} → {days[-1]})  min ₹{y.min():,.0f}  max ₹{y.max():,.0f}")

## Baselines — the numbers to beat

In [ ]:
def mape(a, p):
    return float(np.mean(np.abs((a - p) / a)) * 100)

TEST, STEP = 90, 7
origins = list(range(len(y) - TEST, len(y), STEP))
print(f"{len(origins)} origins, refit every {STEP}d, horizon 1d")

def roll(predict):
    """predict(hist) -> next-day forecast; returns (actuals, preds)."""
    a, p = [], []
    for o in origins:
        a.append(y[o])
        p.append(predict(y[:o]))
    return np.array(a), np.array(p)

base = {}
for name, fn in [("naive", lambda h: h[-1]),
                   ("seasonal-naive-7", lambda h: h[-7]),
                   ("ma-7", lambda h: h[-7:].mean())]:
    a, p = roll(fn)
    base[name] = (mape(a, p), float(np.sqrt(np.mean((a - p) ** 2))), float(np.mean(np.abs(a - p))))
    print(f"{name}: MAPE={base[name][0]:.2f}% RMSE=₹{base[name][1]:,.0f} MAE=₹{base[name][2]:,.0f}")

## Holt-Winters — additive trend + weekly season, grid-fit per origin

In [ ]:
def hw_fit(h, m=7):
    """Grid-search (a,b,g); returns one-step forecast. Pure numpy, additive form."""
    best_sse, best_cfg = float("inf"), None
    for a in (0.1, 0.3, 0.6):
        for b in (0.05, 0.2):
            for g in (0.1, 0.3):
                l, t = h[:m].mean(), (h[m:2*m].mean() - h[:m].mean()) / m
                s = h[:m] - l
                sse = 0.0
                for i in range(m, len(h)):
                    f = l + t + s[i % m]
                    sse += (h[i] - f) ** 2
                    nl = a * (h[i] - s[i % m]) + (1 - a) * (l + t)
                    nt = b * (nl - l) + (1 - b) * t
                    s[i % m] = g * (h[i] - nl) + (1 - g) * s[i % m]
                    l, t = nl, nt
                if sse < best_sse:
                    best_sse, best_cfg = float(sse), (a, b, g, l, t, s.copy())
    a, b, g, l, t, s = best_cfg
    return l + t + s[len(h) % m]

a, p = roll(hw_fit)
base["holt-winters"] = (mape(a, p), float(np.sqrt(np.mean((a - p) ** 2))), float(np.mean(np.abs(a - p))))
print(f"holt-winters: MAPE={base['holt-winters'][0]:.2f}% RMSE=₹{base['holt-winters'][1]:,.0f}")

## ML / DL — lag + calendar features, same origins, same metric

In [ ]:
import time, warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

idx = pd.DatetimeIndex(days)
LAG = [1, 2, 3, 7, 14, 21, 28]

def frame(hist, end):
    """Feature rows for hist[:end]; PAST ONLY (all windows end before the target day)."""
    s = pd.Series(hist)
    X, t = [], []
    for i in range(28, end):
        w = s.iloc[:i]
        row = [w.iloc[-L] for L in LAG] + [w.iloc[-7:].mean(), w.iloc[-28:].mean(),
               w.iloc[-7:].std(ddof=0), float(i)]
        d = idx[i]
        row += [d.dayofweek, d.month, d.dayofyear]
        X.append(row); t.append(s.iloc[i])
    return np.array(X), np.array(t)

FEATS = [f"lag{L}" for L in LAG] + ["roll7", "roll28", "std7", "t", "dow", "month", "doy"]

def roll_ml(make, scale=False):
    a, p = [], []
    for o in origins:
        X, t = frame(y[:o], o)
        Xo, _ = frame(y[:o + 1], o + 1)
        x_next = Xo[-1:]
        model = make()
        if scale:  # features AND target (₹1e7 scale diverges unscaled nets)
            sc, scy = StandardScaler().fit(X), StandardScaler().fit(t.reshape(-1, 1))
            X, x_next, t = sc.transform(X), sc.transform(x_next), scy.transform(t.reshape(-1, 1)).ravel()
            model.fit(X, t)
            pred = scy.inverse_transform(model.predict(x_next).reshape(-1, 1))[0, 0]
        else:
            model.fit(X, t)
            pred = float(model.predict(x_next)[0])
        a.append(y[o]); p.append(pred)
    return np.array(a), np.array(p)

t0 = time.time()
a, p = roll_ml(lambda: RandomForestRegressor(n_estimators=200, min_samples_leaf=10, n_jobs=-1, random_state=42))
base["random-forest"] = (mape(a, p), 0.0, 0.0); print(f"random-forest: MAPE={base['random-forest'][0]:.2f}% ({time.time()-t0:.0f}s)")
t0 = time.time()
a, p = roll_ml(lambda: GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42))
base["gbm"] = (mape(a, p), 0.0, 0.0); print(f"gbm: MAPE={base['gbm'][0]:.2f}% ({time.time()-t0:.0f}s)")
t0 = time.time()
a, p = roll_ml(lambda: MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42), scale=True)
base["mlp(100-50)"] = (mape(a, p), 0.0, 0.0); print(f"mlp: MAPE={base['mlp(100-50)'][0]:.2f}% ({time.time()-t0:.0f}s)")

## PyTorch MLP — same features, full manual loop

Same protocol, same features, standardized X and y, Adam/MSE. Removes any doubt that the sklearn-MLP failure was an implementation artifact.

In [ ]:
import torch
import torch.nn as nn

def roll_torch(seed=42):
    torch.manual_seed(seed)
    a, p = [], []
    for o in origins:
        X, t = frame(y[:o], o)
        Xo, _ = frame(y[:o + 1], o + 1)
        x_next = Xo[-1:]
        sc = StandardScaler().fit(X)
        scy = StandardScaler().fit(t.reshape(-1, 1))
        Xs = torch.tensor(sc.transform(X), dtype=torch.float32)
        ys = torch.tensor(scy.transform(t.reshape(-1, 1)), dtype=torch.float32)
        xn = torch.tensor(sc.transform(x_next), dtype=torch.float32)
        net = nn.Sequential(nn.Linear(Xs.shape[1], 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
        opt = torch.optim.Adam(net.parameters(), lr=1e-3)
        loss = nn.MSELoss()
        net.train()
        for _ in range(300):
            opt.zero_grad(); l = loss(net(Xs), ys); l.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            pred = scy.inverse_transform(net(xn).numpy())[0, 0]
        a.append(y[o]); p.append(float(pred))
    return np.array(a), np.array(p)

t0 = time.time()
a, p = roll_torch()
base["torch-mlp"] = (mape(a, p), 0.0, 0.0)
print(f"torch-mlp: MAPE={base['torch-mlp'][0]:.2f}% ({time.time()-t0:.0f}s)")


## Prophet — additive trend + weekly/yearly season, per origin

The industry default for business time series. Same rolling protocol; MAP decides on the same MAPE.

In [ ]:
import logging
logging.getLogger("cmdstanpy").disabled = True
logging.getLogger("prophet").disabled = True
from prophet import Prophet

def roll_prophet():
    a, p = [], []
    for o in origins:
        hist = pd.DataFrame({"ds": idx[:o], "y": y[:o]})
        m = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
        m.fit(hist)
        fc = m.predict(m.make_future_dataframe(periods=1))
        a.append(y[o]); p.append(float(fc["yhat"].iloc[-1]))
    return np.array(a), np.array(p)

t0 = time.time()
a, p = roll_prophet()
base["prophet"] = (mape(a, p), 0.0, 0.0)
print(f"prophet: MAPE={base['prophet'][0]:.2f}% ({time.time()-t0:.0f}s)")


## Scoreboard + what the winner gets wrong

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print(f"{'model':16s} {'MAPE':>7s}")
for name, (m, _, _) in sorted(base.items(), key=lambda kv: kv[1][0]):
    print(f"{name:16s} {m:6.2f}%")

# Re-run the winner for the plot (actuals vs preds on the origin dates)
a, p = roll(hw_fit)
_, p_naive = roll(lambda h: h[-7])
test_days = [days[o] for o in origins]
plt.figure(figsize=(9, 3.5))
plt.plot(test_days, a / 1e6, label="actual", linewidth=1.5)
plt.plot(test_days, p / 1e6, label="holt-winters", linewidth=1.2)
plt.plot(test_days, p_naive / 1e6, label="seasonal-naive", linewidth=1, alpha=0.7)
plt.xticks(test_days[::15], rotation=30, fontsize=8); plt.ylabel("revenue (₹M)"); plt.legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/opencode/nb04_forecast.png"); print("plot: /tmp/opencode/nb04_forecast.png")

## Verdict (measured 2026-09-12, rolling 90d, 13 origins)

| # | Model | MAPE | Note |
|---|---|---|---|
| 1 | **RandomForest** | **3.38%** | winner; lags + calendar generalize |
| 2 | Prophet | 3.96% | best explainable (trend + weekly/yearly season) |
| 3 | Holt-Winters | 4.27% | transparent, near-free |
| 4 | torch-MLP | 4.40% | runs correctly, no edge at 731 points |
| 5 | GBM | 4.60% | ≈ moving average — trees can't extrapolate trend |
| 6 | MA-7 | 4.66% | the cheap baseline to beat |
| 7 | sklearn-MLP | 4.67% | same story as torch |
| 8 | Naive | 5.13% | |
| 9 | Seasonal-naive-7 | 5.86% | weekly season is weak — trend dominates |

**Decision: ship RandomForest** (42% error reduction vs seasonal-naive, single-digit MAPE). Prophet stays as the interpretable challenger. DL (both MLPs) is honestly mid-pack — kept in the comparison, not in production.

*Extra deps for this notebook only (experiment-scoped, not product deps):*
`uv pip install --index-url https://download.pytorch.org/whl/cpu torch` · `uv pip install prophet`
